<a href="https://colab.research.google.com/github/Kieunhungtruong/Econometrics/blob/main/pythonversion/chapter7.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<div style="display:flex;flex-direction:column;align-items:center;justify-content:center;gap:10px;">
  <h1 style="text-align:center;font-size:26px;font-weight:bold;font-family:'Nunito';color:purple;">
    Chapter 7 – Số liệu bảng (Panel Data)
  </h1>
</div>

## Kết nối Google Drive và cài gói cần thiết

In [2]:
from google.colab import drive
drive.mount("/content/drive/", force_remount=True)

Mounted at /content/drive/


In [3]:
!pip install pyreadstat linearmodels -q

import pandas as pd
import numpy as np
import statsmodels.api as sm
import matplotlib.pyplot as plt
from linearmodels.panel import PanelOLS, RandomEffects, PooledOLS
from linearmodels.panel.model import BetweenOLS
from scipy.stats import chi2
import pyreadstat
import warnings
warnings.filterwarnings('ignore')

palette = ["#53b0ae", "#a31414", "#2b6999", "#e37000", "#b2c615", "#88837d", "#B3B3B3"]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 30.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 62.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.3/117.3 kB 8.3 MB/s eta 0:00:00


## Đọc dữ liệu

In [4]:
df, meta = pyreadstat.read_dta(
    "/content/drive/MyDrive/econometrics/rawdata/panel121416.dta",
    encoding="latin1",
)
df.head()

,year,hhid,HHsize,agehead,genderhead,female_ratio,migration,remittances,num_eduhead,per_expenditure,children_ratio,elderly_ratio,ln_per_expenditure,agehead2
0,2012.0,32.0,5.0,59.0,1.0,0.4,0.0,10000.0,12,66575.000000,0.4,0.0,11.106085,3481.0
1,2014.0,32.0,2.0,61.0,1.0,0.5,1.0,19000.0,12,51281.000000,0.0,0.5,10.845076,3721.0
2,2016.0,32.0,2.0,63.0,1.0,0.5,0.0,60000.0,12,57732.000000,0.0,0.5,10.963567,3969.0
3,2012.0,33.0,5.0,33.0,0.0,0.8,0.0,130000.0,12,59666.199219,0.4,0.0,10.996521,1089.0
4,2014.0,33.0,5.0,35.0,0.0,0.6,0.0,253000.0,12,85886.601562,0.4,0.0,11.360784,1225.0


---
# 7.1 Mô hình hồi quy với số liệu bảng

Mô hình hồi quy với **dữ liệu bảng** (panel data) dựa vào số liệu ở nhiều đối tượng với nhiều đặc điểm được quan sát nhiều lần qua thời gian.

**Mô hình tổng quát:**

$$y_{it} = \beta_0 + \beta_1 x_{1,it} + \beta_2 x_{2,it} + \cdots + \beta_k x_{k,it} + w_{it}$$

Trong đó $i$ là đơn vị quan sát (hộ gia đình, cá nhân, công ty, ...), $t$ là thời kỳ.

Phần sai số gộp: $w_{it} = u_i + \varepsilon_{it}$

**Ưu điểm của dữ liệu bảng:**
1. Làm tăng số quan sát có trong mẫu.
2. Có thể nghiên cứu được sự thay đổi động (dynamic change) qua thời gian.
3. Có thể nghiên cứu các mô hình hành vi phức tạp, bao gồm các biến không thay đổi qua thời gian.

**Các phương pháp ước lượng phổ biến:**
- **Pooled OLS**: bỏ qua yếu tố thời gian và sự khác biệt giữa các đơn vị.
- **FEM – Fixed Effects Model (tác động cố định)**: kiểm soát tính riêng biệt của từng đơn vị.
- **REM – Random Effects Model (tác động ngẫu nhiên)**: giả định hệ số chặn của từng đơn vị là ngẫu nhiên từ một tổng thể lớn hơn.

---
# 7.2 Mô hình tác động cố định (Fixed Effects Model – FEM)

Trong FEM, hệ số chặn được phép khác nhau giữa các cá nhân để phản ánh đặc điểm riêng của từng đơn vị.

**Phương pháp Within-group (bên trong nhóm):** Trừ giá trị trung bình của từng đơn vị ra khỏi mỗi biến:

$$\tilde{y}_{it} = \tilde{x}_{1,it}\beta_1 + \tilde{x}_{2,it}\beta_2 + \cdots + \tilde{x}_{k,it}\beta_k + \tilde{\varepsilon}_{it}$$

Trong đó $\tilde{y}_{it} = y_{it} - \bar{y}_i$, $\tilde{x}_{j,it} = x_{j,it} - \bar{x}_{j,i}$.

> **Lưu ý:** Phương pháp within-group loại bỏ các biến không thay đổi theo thời gian (time-invariant) như giới tính, chủng tộc,...

---
# 7.4 Mô hình các tác động ngẫu nhiên (Random Effects Model – REM)

Trong REM, hệ số chặn của từng cá nhân được giả định là rút ngẫu nhiên từ một tổng thể lớn hơn và có giá trị trung bình không đổi.

REM sử dụng phép biến đổi quasi-differencing:

$$\tilde{y}_{it} = y_{it} - \lambda \bar{y}_i$$

Trong đó $\lambda \in [0, 1]$:
- $\lambda = 0$: REM tương đương Pooled OLS.
- $\lambda = 1$: REM tương đương FEM.

**Hệ số tương quan nội nhóm (rho):**

$$\rho = \frac{\sigma_u^2}{\sigma_u^2 + \sigma_e^2}$$

Khi $\rho$ gần 1, thành phần sai số chủ yếu đến từ hiệu ứng cá nhân $u_i$, nên FEM phù hợp hơn.

---
# 7.5 Nên sử dụng Fixed Effects hay Random Effects?

| Kiểm định | Lựa chọn |
|-----------|----------|
| **F-test** (kiểm định tác động cố định) | Nếu p < 0.05 → FEM phù hợp hơn Pooled OLS |
| **Breusch-Pagan LM test** | Nếu p < 0.05 → REM phù hợp hơn Pooled OLS |
| **Kiểm định Hausman** | Nếu p < 0.05 → FEM phù hợp hơn REM |

**Kiểm định Hausman:**
- $H_0$: FEM và REM không có khác biệt đáng kể (REM phù hợp).
- $H_1$: Có sự khác biệt hệ thống giữa hai mô hình (FEM phù hợp).
- Thống kê kiểm định: $\chi^2$ với bậc tự do bằng số biến giải thích.

---
# 7.7 Ví dụ: Tác động của di cư đến chi tiêu mỗi người trong hộ

Dữ liệu bảng gồm **5,736 quan sát** ở các năm 2012, 2014 và 2016, thu thập từ Khảo sát Mức sống Hộ gia đình Việt Nam (VHLSS).

- **Dữ liệu cân bằng (balanced panel):** mỗi năm có 1,912 hộ, mỗi hộ có dữ liệu trải dài cả ba năm.
- **Dữ liệu bảng ngắn (short panel):** N (số hộ) > T (số thời kỳ).

**Biến phụ thuộc:** `ln_per_expenditure` – log của chi tiêu bình quân mỗi người trong hộ.

**Biến độc lập (`xvar`):**
- `HHsize`: Quy mô hộ gia đình
- `agehead`: Tuổi của chủ hộ
- `children_ratio`: Tỷ lệ trẻ em trong hộ
- `elderly_ratio`: Tỷ lệ người cao tuổi trong hộ
- `genderhead`: Giới tính chủ hộ (1 = nữ)
- `female_ratio`: Tỷ lệ nữ trong hộ
- `agehead2`: Tuổi chủ hộ bình phương
- `num_eduhead`: Số năm đi học của chủ hộ
- `migration`: Có người di cư trong hộ (1 = có)

## Chuẩn bị dữ liệu

In [5]:
# Tạo biến agehead2 và id nếu chưa có
df["agehead2"] = df["agehead"] ** 2

# Tạo biến giả năm
df["_2012"] = (df["year"] == 2012).astype(int)
df["_2014"] = (df["year"] == 2014).astype(int)

# Tạo id nhóm từ hhid
df["id"] = df.groupby("hhid").ngroup() + 1

xvar = ["HHsize", "agehead", "children_ratio", "elderly_ratio",
        "genderhead", "female_ratio", "agehead2", "num_eduhead", "migration"]

print(f"Số quan sát: {len(df):,}")
print(f"Số hộ (N): {df['id'].nunique():,}")
print(f"Số năm (T): {df['year'].nunique()}")
print(f"Năm: {sorted(df['year'].unique())}")
print("\nThống kê mô tả:")
df[["ln_per_expenditure"] + xvar].describe().round(4)

Số quan sát: 5,736
Số hộ (N): 1,912
Số năm (T): 3
Năm: [np.float64(2012.0), np.float64(2014.0), np.float64(2016.0)]

Thống kê mô tả:


,ln_per_expenditure,HHsize,agehead,children_ratio,elderly_ratio,genderhead,female_ratio,agehead2,num_eduhead,migration
count,5736.0000,5736.0000,5736.0000,5736.0000,5736.0000,5736.0000,5736.0000,5736.0000,5736.0000,5736.0000
mean,9.8442,3.8161,51.5938,0.1944,0.1598,0.2422,0.5255,2854.0631,7.1979,0.1219
std,0.6296,1.5540,13.8628,0.2026,0.2958,0.4284,0.2087,1544.2790,3.7114,0.3272
min,7.8021,1.0000,16.0000,0.0000,0.0000,0.0000,0.0000,256.0000,0.0000,0.0000
25%,9.4252,3.0000,42.0000,0.0000,0.0000,0.0000,0.4000,1764.0000,4.0000,0.0000
50%,9.8447,4.0000,50.0000,0.2000,0.0000,0.0000,0.5000,2500.0000,8.0000,0.0000
75%,10.2448,5.0000,60.0000,0.3333,0.2000,0.0000,0.6667,3600.0000,10.0000,0.0000
max,14.0620,13.0000,99.0000,0.7500,1.0000,1.0000,1.0000,9801.0000,12.0000,1.0000


---
# Mô hình Pooled OLS về tác động của di cư (Bảng 7.1)

Mô hình OLS gộp đơn giản hóa dữ liệu bảng bằng cách bỏ qua yếu tố thời gian:

$$\ln\_per\_expenditure_i = \beta_0 + \beta_1 HHsize_i + \cdots + \beta_9 migration_i + \varepsilon_i$$

**Hạn chế:** Pooled OLS bỏ qua tính riêng biệt của từng hộ, khiến sai số có thể tương quan với các biến độc lập, vi phạm giả định BLUE.

In [6]:
X = sm.add_constant(df[xvar])
y = df["ln_per_expenditure"]

model_pooled = sm.OLS(y, X).fit()
print("=== Mô hình Pooled OLS (Bảng 7.1) ===")
print(model_pooled.summary())

=== Mô hình Pooled OLS (Bảng 7.1) ===
                            OLS Regression Results                            
Dep. Variable:     ln_per_expenditure   R-squared:                       0.281
Model:                            OLS   Adj. R-squared:                  0.280
Method:                 Least Squares   F-statistic:                     249.1
Date:                Wed, 08 Apr 2026   Prob (F-statistic):               0.00
Time:                        11:42:19   Log-Likelihood:                -4537.1
No. Observations:                5736   AIC:                             9094.
Df Residuals:                    5726   BIC:                             9161.
Df Model:                           9                                         
Covariance Type:            nonrobust                                         
                     coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------
const 

## Mô hình Pooled OLS với biến giả năm (Bảng 7.2)

Thêm biến giả năm để kiểm soát tác động của thời gian:

$$\ln\_per\_expenditure_i = \beta_0 + \cdots + \beta_9 migration_i + \beta_{10} \_2012_i + \beta_{11} \_2014_i + \varepsilon_i$$

Năm 2016 là nhóm tham chiếu.

In [7]:
xvar_year = xvar + ["_2012", "_2014"]
X_year = sm.add_constant(df[xvar_year])

model_pooled_year = sm.OLS(y, X_year).fit()
print("=== Mô hình Pooled OLS với biến giả năm (Bảng 7.2) ===")
print(model_pooled_year.summary())

=== Mô hình Pooled OLS với biến giả năm (Bảng 7.2) ===
                            OLS Regression Results                            
Dep. Variable:     ln_per_expenditure   R-squared:                       0.306
Model:                            OLS   Adj. R-squared:                  0.305
Method:                 Least Squares   F-statistic:                     229.3
Date:                Wed, 08 Apr 2026   Prob (F-statistic):               0.00
Time:                        11:42:19   Log-Likelihood:                -4437.5
No. Observations:                5736   AIC:                             8899.
Df Residuals:                    5724   BIC:                             8979.
Df Model:                          11                                         
Covariance Type:            nonrobust                                         
                     coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------

---
# Mô hình FEM – Fixed Effects (Bảng 7.3)

Ước lượng FEM (Within-group) khắc phục hạn chế của Pooled OLS bằng cách kiểm soát tính riêng biệt của từng hộ.

Phương trình ước lượng sau khi trừ giá trị trung bình:

$$\tilde{y}_{it} = \beta_1 \widetilde{HHsize}_{it} + \cdots + \beta_9 \widetilde{migration}_{it} + \tilde{\varepsilon}_{it}$$

> Hằng số $\beta_0$ bị triệt tiêu vì không chịu tác động theo thời gian.

In [8]:
# Chuẩn bị dữ liệu dạng panel (MultiIndex: entity, time)
df_panel = df.set_index(["id", "year"])

y_panel = df_panel["ln_per_expenditure"]
X_panel = df_panel[xvar]

# FEM – Fixed Effects (Within-group)
fem = PanelOLS(y_panel, X_panel, entity_effects=True, drop_absorbed=True)
fem_res = fem.fit(cov_type='unadjusted')
print("=== Mô hình FEM – Fixed Effects (Bảng 7.3) ===")
print(fem_res)

# ── sigma_u, sigma_e, rho – khớp Stata xtreg,fe ─────────────────────────────
resid   = fem_res.resids.copy()
N       = df_panel.index.get_level_values("id").nunique()
T_total = len(df_panel)
K       = len(xvar)
T_bar   = T_total / N

# sigma_e: RSS_within / (T_total - N - K)
rss_within = (resid**2).sum()
sigma_e2   = rss_within / (T_total - N - K)
sigma_e    = np.sqrt(sigma_e2)

# alpha_i = entity fixed effects: y_bar_i - X_bar_i @ beta_FEM
y_bar_i = df_panel[y_panel.name].groupby(level=0).mean()
X_bar_i = df_panel[xvar].groupby(level=0).mean()
alpha_i = y_bar_i - X_bar_i @ fem_res.params

# sigma_u: phương sai mẫu của alpha_i (khớp Stata)
sigma_u2 = (alpha_i - alpha_i.mean()).pow(2).sum() / (N - 1)
sigma_u2 = max(sigma_u2, 0)
sigma_u  = np.sqrt(sigma_u2)
rho      = sigma_u2 / (sigma_u2 + sigma_e2)

print(f"sigma_u = {sigma_u:.8f}")
print(f"sigma_e = {sigma_e:.8f}")
print(f"rho     = {rho:.8f}")

# ── _cons của FEM = grand mean của entity effects (khớp Stata _cons) ──────────
# Stata báo _cons = y_dbar - X_dbar @ beta, SE tính bằng delta method
const_fem   = alpha_i.mean()
X_dbar      = df_panel[xvar].mean()                           # grand means của X
se_const_fem = np.sqrt(X_dbar.values @ fem_res.cov.values @ X_dbar.values)
t_const_fem  = const_fem / se_const_fem

print(f"_cons   = {const_fem:.7f}  SE = {se_const_fem:.7f}  t = {t_const_fem:.3f}")


=== Mô hình FEM – Fixed Effects (Bảng 7.3) ===
                          PanelOLS Estimation Summary                           
Dep. Variable:     ln_per_expenditure   R-squared:                        0.0912
Estimator:                   PanelOLS   R-squared (Between):              0.2130
No. Observations:                5736   R-squared (Within):               0.0912
Date:                Wed, Apr 08 2026   R-squared (Overall):              0.2129
Time:                        11:42:20   Log-likelihood                   -1140.4
Cov. Estimator:            Unadjusted                                           
                                        F-statistic:                      42.541
Entities:                        1912   P-value                           0.0000
Avg Obs:                       3.0000   Distribution:                  F(9,3815)
Min Obs:                       3.0000                                           
Max Obs:                       3.0000   F-statistic (robust): 

## FEM với Sai số chuẩn mạnh – Robust Standard Errors (Bảng 7.4)

Sử dụng phương pháp Huber-White để giảm tác động của hiện tượng phương sai thay đổi trong FEM.

In [9]:
fem_robust = PanelOLS(y_panel, X_panel, entity_effects=True, drop_absorbed=True)
fem_robust_res = fem_robust.fit(cov_type='clustered', cluster_entity=True)
print("=== FEM với Sai số chuẩn mạnh (Bảng 7.4) ===")
print(fem_robust_res)

# sigma_u, sigma_e, rho giống FEM (chỉ khác std.err) – dùng lại từ Bảng 7.3
print(f"sigma_u = {sigma_u:.8f}")
print(f"sigma_e = {sigma_e:.8f}")
print(f"rho     = {rho:.8f}")


=== FEM với Sai số chuẩn mạnh (Bảng 7.4) ===
                          PanelOLS Estimation Summary                           
Dep. Variable:     ln_per_expenditure   R-squared:                        0.0912
Estimator:                   PanelOLS   R-squared (Between):              0.2130
No. Observations:                5736   R-squared (Within):               0.0912
Date:                Wed, Apr 08 2026   R-squared (Overall):              0.2129
Time:                        11:42:20   Log-likelihood                   -1140.4
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      42.541
Entities:                        1912   P-value                           0.0000
Avg Obs:                       3.0000   Distribution:                  F(9,3815)
Min Obs:                       3.0000                                           
Max Obs:                       3.0000   F-statistic (robust):   

---
# Mô hình REM – Random Effects (Bảng 7.5)

REM giả định các hệ số chặn là ngẫu nhiên, được rút ra từ một tổng thể lớn hơn.

**Hệ số tương quan nội nhóm (rho):**

$$\rho = \frac{\sigma_u^2}{\sigma_u^2 + \sigma_e^2}$$

Nếu $\rho$ lớn → phần lớn biến thiên đến từ hiệu ứng cố định → FEM phù hợp hơn.

In [10]:
# REM yêu cầu thêm hằng số (const) vào X – Stata tự thêm, linearmodels thì không
X_panel_rem = sm.add_constant(df_panel[xvar])

rem = RandomEffects(y_panel, X_panel_rem)
rem_res = rem.fit(cov_type='unadjusted')
print("=== Mô hình REM – Random Effects (Bảng 7.5) ===")
print(rem_res)

# ── sigma_u, sigma_e, rho – khớp Stata xtreg (REM) ─────────────────────────
# linearmodels lưu variance components trong variance_decomposition:
#   Effects  = sigma_u^2  (between variance)
#   Residual = sigma_e^2  (within/idiosyncratic variance)
var_decomp   = rem_res.variance_decomposition
sigma_u2_rem = float(var_decomp.loc["Effects"])
sigma_e2_rem = float(var_decomp.loc["Residual"])
sigma_u_rem  = np.sqrt(sigma_u2_rem)
sigma_e_rem  = np.sqrt(sigma_e2_rem)
rho_rem      = sigma_u2_rem / (sigma_u2_rem + sigma_e2_rem)

print(f"sigma_u = {sigma_u_rem:.8f}")
print(f"sigma_e = {sigma_e_rem:.8f}")
print(f"rho     = {rho_rem:.8f}")


=== Mô hình REM – Random Effects (Bảng 7.5) ===
                        RandomEffects Estimation Summary                        
Dep. Variable:     ln_per_expenditure   R-squared:                        0.1815
Estimator:              RandomEffects   R-squared (Between):              0.3419
No. Observations:                5736   R-squared (Within):               0.0686
Date:                Wed, Apr 08 2026   R-squared (Overall):              0.2758
Time:                        11:42:21   Log-likelihood                   -2368.6
Cov. Estimator:            Unadjusted                                           
                                        F-statistic:                      141.06
Entities:                        1912   P-value                           0.0000
Avg Obs:                       3.0000   Distribution:                  F(9,5726)
Min Obs:                       3.0000                                           
Max Obs:                       3.0000   F-statistic (robust):

## REM với Sai số chuẩn mạnh (Bảng 7.6)

In [11]:
rem_robust_res = rem.fit(cov_type='clustered', cluster_entity=True)
print("=== REM với Sai số chuẩn mạnh (Bảng 7.6) ===")
print(rem_robust_res)

# sigma_u, sigma_e, rho giống REM (chỉ khác std.err) – dùng lại từ Bảng 7.5
print(f"sigma_u = {sigma_u_rem:.8f}")
print(f"sigma_e = {sigma_e_rem:.8f}")
print(f"rho     = {rho_rem:.8f}")


=== REM với Sai số chuẩn mạnh (Bảng 7.6) ===
                        RandomEffects Estimation Summary                        
Dep. Variable:     ln_per_expenditure   R-squared:                        0.1815
Estimator:              RandomEffects   R-squared (Between):              0.3419
No. Observations:                5736   R-squared (Within):               0.0686
Date:                Wed, Apr 08 2026   R-squared (Overall):              0.2758
Time:                        11:42:21   Log-likelihood                   -2368.6
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      141.06
Entities:                        1912   P-value                           0.0000
Avg Obs:                       3.0000   Distribution:                  F(9,5726)
Min Obs:                       3.0000                                           
Max Obs:                       3.0000   F-statistic (robust):   

---
# So sánh mô hình Pooled OLS, FEM và REM (Bảng 7.8)

## Kiểm định F trong lựa chọn Pooled OLS và FEM

Kết quả kiểm định F trong FEM:
- $H_0$: Không có tác động cố định (Pooled OLS phù hợp).
- Nếu p < 0.05 → FEM phù hợp hơn Pooled OLS.

## Kiểm định Breusch-Pagan LM trong lựa chọn Pooled OLS và REM

- $H_0$: $\text{Var}(u) = 0$ → Pooled OLS phù hợp.
- $H_1$: $\text{Var}(u) > 0$ → REM phù hợp.

## Kiểm định Hausman trong lựa chọn FEM và REM

- $H_0$: Sự khác biệt giữa FEM và REM không có hệ thống → REM phù hợp.
- Nếu p < 0.05 → FEM phù hợp hơn REM.

In [12]:
# Kiểm định Breusch-Pagan LM (tương đương xttest0 trong Stata)
# Dùng statsmodels với mô hình RE từ linearmodels

# Tính LM test thủ công
# residuals từ Pooled OLS, kiểm định tương quan nội nhóm
df_panel_copy = df.copy()
df_panel_copy["resid_pooled"] = model_pooled.resid.values

# Tính tổng bình phương phần dư trong nhóm (between) và tổng thể
group_mean_resid = df_panel_copy.groupby("id")["resid_pooled"].mean()
df_panel_copy["group_mean_resid"] = df_panel_copy["id"].map(group_mean_resid)

n = len(df_panel_copy)
N = df_panel_copy["id"].nunique()   # số nhóm
T = df_panel_copy.groupby("id").size().mean()  # số quan sát trung bình mỗi nhóm

sum_group_sq = (df_panel_copy.groupby("id")["resid_pooled"].sum() ** 2).sum()
sum_total_sq = (df_panel_copy["resid_pooled"] ** 2).sum()

LM = (n / (2 * (T - 1))) * (sum_group_sq / sum_total_sq - 1) ** 2
p_lm = 1 - chi2.cdf(LM, df=1)

print("=== Kiểm định Breusch-Pagan LM (tương đương xttest0) ===")
print(f"LM statistic (chibar2):  {LM:.4f}")
print(f"P-value (Prob > chibar2): {p_lm:.4f}")
if p_lm < 0.05:
    print("→ Bác bỏ H₀: REM phù hợp hơn Pooled OLS")
else:
    print("→ Không bác bỏ H₀: Pooled OLS phù hợp")

=== Kiểm định Breusch-Pagan LM (tương đương xttest0) ===
LM statistic (chibar2):  1560.6864
P-value (Prob > chibar2): 0.0000
→ Bác bỏ H₀: REM phù hợp hơn Pooled OLS


In [13]:
# Kiểm định Hausman – FEM vs REM
# FEM không có const; REM có const → chỉ so sánh các biến chung

b_fem = fem_res.params          # Hệ số FEM (nhất quán)
b_rem = rem_res.params          # Hệ số REM (hiệu quả dưới H0)

# Chỉ lấy các biến xuất hiện trong cả hai (loại bỏ const)
common_vars = b_fem.index.intersection(b_rem.index)
b_diff = b_fem[common_vars] - b_rem[common_vars]

V_fem = fem_res.cov.loc[common_vars, common_vars]
V_rem = rem_res.cov.loc[common_vars, common_vars]
V_diff = V_fem - V_rem

# Thống kê kiểm định Hausman
try:
    chi2_stat = float(b_diff.values @ np.linalg.pinv(V_diff.values) @ b_diff.values)
    df_hausman = len(common_vars)
    p_hausman = 1 - chi2.cdf(chi2_stat, df=df_hausman)

    print("=== Kiểm định Hausman – FEM vs REM (Bảng 7.7) ===")
    print(f"\n{'Biến':20s} {'FEM (b)':>12s} {'REM (B)':>12s} {'Difference':>12s}")
    print("-" * 60)
    for var in common_vars:
        print(f"{var:20s} {b_fem[var]:12.7f} {b_rem[var]:12.7f} {b_diff[var]:12.7f}")
    print("-" * 60)
    print(f"\nchi2({df_hausman}) = {chi2_stat:.2f}")
    print(f"Prob > chi2 = {p_hausman:.4f}")
    if p_hausman < 0.05:
        print("→ Bác bỏ H₀: FEM phù hợp hơn REM")
    else:
        print("→ Không bác bỏ H₀: REM phù hợp")
except Exception as e:
    print(f"Lỗi tính Hausman: {e}")


=== Kiểm định Hausman – FEM vs REM (Bảng 7.7) ===

Biến                      FEM (b)      REM (B)   Difference
------------------------------------------------------------
HHsize                 -0.0546127   -0.0666961    0.0120834
agehead                 0.0360423    0.0360040    0.0000383
children_ratio         -0.3739499   -0.3196301   -0.0543198
elderly_ratio          -0.0710151   -0.2066921    0.1356770
genderhead              0.1423765    0.1315705    0.0108061
female_ratio           -0.0576705   -0.1504372    0.0927667
agehead2               -0.0002424   -0.0002831    0.0000408
num_eduhead             0.0299362    0.0598276   -0.0298915
migration               0.1370016    0.1237871    0.0132145
------------------------------------------------------------

chi2(9) = 140.12
Prob > chi2 = 0.0000
→ Bác bỏ H₀: FEM phù hợp hơn REM


## Bảng so sánh kết quả (Bảng 7.8)

In [16]:
# Bảng so sánh Pooled OLS, FEM, REM
pooled_params = model_pooled.params
fem_params    = fem_res.params.copy()
rem_params    = rem_res.params

# Thêm _cons vào FEM (grand mean của entity effects)
fem_params["const"] = const_fem

all_vars = pooled_params.index.tolist()   # const + 9 biến
comparison = pd.DataFrame(index=all_vars)
comparison["Pooled OLS"] = pooled_params
comparison["FEM"]        = fem_params.reindex(all_vars)
comparison["REM"]        = rem_params.reindex(all_vars)

# t/z statistics
pooled_t = model_pooled.tvalues
fem_t    = fem_res.tstats.copy()
fem_t["const"] = t_const_fem
rem_z    = rem_res.tstats

tstat = pd.DataFrame(index=all_vars)
tstat["Pooled OLS (t)"] = pooled_t
tstat["FEM (t)"]        = fem_t.reindex(all_vars)
tstat["REM (z)"]        = rem_z.reindex(all_vars)

print("=== Hệ số ước lượng ===")
print(comparison.round(6).to_string())
print("=== Thống kê t/z ===")
print(tstat.round(3).to_string())

=== Hệ số ước lượng ===
                Pooled OLS       FEM       REM
const             8.777095  8.732469  8.745807
HHsize           -0.072745 -0.054613 -0.066696
agehead           0.034190  0.036042  0.036004
children_ratio   -0.303980 -0.373950 -0.319630
elderly_ratio    -0.271504 -0.071015 -0.206692
genderhead        0.130080  0.142377  0.131570
female_ratio     -0.188046 -0.057670 -0.150437
agehead2         -0.000267 -0.000242 -0.000283
num_eduhead       0.069775  0.029936  0.059828
migration         0.095824  0.137002  0.123787
=== Thống kê t/z ===
                Pooled OLS (t)  FEM (t)  REM (z)
const                   86.934   43.668   73.125
HHsize                 -12.826   -6.393  -10.708
agehead                  9.355    5.154    8.370
children_ratio          -6.432   -5.827   -6.453
elderly_ratio           -7.289   -1.386   -5.249
genderhead               6.829    3.253    5.505
female_ratio            -4.968   -0.944   -3.489
agehead2                -8.002   -3.845   -7.2

---
# Nhận xét và diễn giải kết quả

## Pooled OLS
- Mô hình giải thích được **28.1%** biến phụ thuộc ($R^2 = 0.2814$).
- Biến `migration` có hệ số **0.0958** (p < 0.001): hộ có người di cư có chi tiêu bình quân cao hơn khoảng **9.58%**, giữ các yếu tố khác không đổi.
- **Hạn chế:** Bỏ qua tính riêng biệt của từng hộ → ước lượng có thể bị lệch.

## FEM (Within-group)
- Kiểm soát được các hiệu ứng cố định riêng của từng hộ.
- Biến `migration` có hệ số **0.137** (p < 0.001): hộ có người di cư có chi tiêu bình quân cao hơn khoảng **13.7%**.
- Kiểm định F cho tác động cố định: **p < 0.001** → FEM phù hợp hơn Pooled OLS.

## REM
- Biến `migration` có hệ số **0.1238** (p < 0.001): hộ có người di cư có chi tiêu bình quân cao hơn **12.4%**.
- $\rho = 0.535$ → khoảng 53.5% biến thiên đến từ hiệu ứng cá nhân $u_i$.

## Kết luận từ kiểm định Hausman (Bảng 7.7)
- $\chi^2(9) = 140.12$, p < 0.001 → **FEM phù hợp hơn REM**.
- Nguyên nhân: sai số ngẫu nhiên có khả năng tương quan với các biến giải thích trong mô hình.

> **Kết luận chung:** Mô hình FEM (với sai số chuẩn mạnh như Bảng 7.4) là mô hình phù hợp nhất để phân tích tác động của di cư đến chi tiêu bình quân mỗi người trong hộ.

<h2 style="font-size: 26px; font-weight: bold; font-family:'Nunito'; color: purple;">About the Authors:</h2>

<a href="https://www.linkedin.com/in/truongnhung2002"> Nhung Truong (Kristen Zhang) </a> has a degree in Investment Economics from UEH, with a focus on quantitative research.

### <h3 align="center"> © 2026 Nhung Truong. Licensed under CC BY 4.0 </h3>